# Day 2: CLIP Embedding Generation

**Goal**: Generate CLIP embeddings for all images and save to disk.

**Done checkpoint**:
- `data/embeddings/embeddings.npy` shape is `(N, 512)`
- `data/embeddings/id_map.json` maps FAISS index positions to image IDs

**Runtime**: Use GPU (T4). ~44k images takes ~15-20 min on T4.

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/PinterestDemo'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'CWD: {os.getcwd()}')

In [ ]:
!pip install -q torch torchvision transformers Pillow pandas numpy tqdm pyyaml
print('Done.')

## 2. Verify GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

## 3. Load Config

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

# Override paths to absolute Drive paths
BASE = Path(PROJECT_DIR)
config['paths']['metadata_csv']    = str(BASE / 'data/metadata/images.csv')
config['paths']['embeddings_dir']  = str(BASE / 'data/embeddings')
config['paths']['embeddings_path'] = str(BASE / 'data/embeddings/embeddings.npy')
config['paths']['id_map_path']     = str(BASE / 'data/embeddings/id_map.json')
config['model']['device']          = 'cuda'  # Use GPU
config['model']['batch_size']      = 128     # T4 can handle 128

print('Config loaded.')
print(f'  Model:      {config["model"]["clip_model"]}')
print(f'  Device:     {config["model"]["device"]}')
print(f'  Batch size: {config["model"]["batch_size"]}')

## 4. Check If Already Done (Resume Support)

In [ ]:
import numpy as np
import json
from pathlib import Path

emb_path = Path(config['paths']['embeddings_path'])
id_map_path = Path(config['paths']['id_map_path'])

if emb_path.exists() and id_map_path.exists():
    existing = np.load(emb_path)
    with open(id_map_path) as f:
        existing_map = json.load(f)
    print(f'Embeddings already exist: shape={existing.shape}')
    print(f'ID map entries: {len(existing_map)}')
    print('Run the next cell only if you want to REBUILD from scratch.')
    REBUILD = False
else:
    print('No existing embeddings found. Will build from scratch.')
    REBUILD = True

## 5. Generate Embeddings

In [ ]:
# Set REBUILD = True above to force rebuild
if REBUILD:
    from scripts.build_embeddings import build_embeddings
    build_embeddings(config)
else:
    print('Skipping — embeddings already exist. Set REBUILD=True to regenerate.')

## 6. Inspect Embeddings

In [ ]:
import numpy as np
import json

embeddings = np.load(config['paths']['embeddings_path'])
with open(config['paths']['id_map_path']) as f:
    id_map = json.load(f)

print(f'Embeddings shape:        {embeddings.shape}')
print(f'Dtype:                   {embeddings.dtype}')
print(f'ID map entries:          {len(id_map)}')
print(f'Sample norms (should ~1.0): {np.linalg.norm(embeddings[:5], axis=1).tolist()}')

# Quick sanity: cosine similarity of an embedding with itself
v = embeddings[0]
print(f'Self-similarity of emb[0]: {float(v @ v):.6f} (should be 1.0)')

## ✅ Day 2 Done Checkpoint

In [ ]:
import numpy as np
from pathlib import Path

emb = np.load(config['paths']['embeddings_path'])
assert emb.ndim == 2 and emb.shape[1] == 512, f'Expected (N, 512), got {emb.shape}'
assert emb.dtype == np.float32, f'Expected float32, got {emb.dtype}'
assert Path(config['paths']['id_map_path']).exists(), 'id_map.json missing'

print('Day 2 COMPLETE')
print(f'  Embeddings: {emb.shape[0]:,} images x {emb.shape[1]} dims')
print(f'  Saved at:   {config["paths"]["embeddings_path"]}')